# 07 · Follow-up cohort: exploratory analysis (P1)

Data checks, ECD over time by procedure, Kaplan-Meier survival, visit attendance and the effect of rejection. Code: `data_prep/followup_eda.py`. Same as `python -m recommendation.main followup-eda`; takes a few seconds.

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
from recommendation.data_prep import followup_eda as E

grafts, visits, source = E.load_cohort(P["followup_cohort"])
print("Data source:", source)
pd.DataFrame(E.validate_cohort(grafts, visits))

## 1. Cohort

In [ ]:
s = E.cohort_summary(grafts, visits)
pd.Series({k: v for k, v in s.items() if not isinstance(v, (dict, list))})

## 2. ECD over time by procedure (median, cells/mm²; month 0 = donor)

In [ ]:
trend = E.measures_over_time(grafts, visits)
display(trend.pivot(index="months_after_surgery", columns="procedure", values="ecd_median").round(0))
res = E.run(P["followup_cohort"], OUTPUTS / "followup_eda")       # writes the figures and summary
display(Image(OUTPUTS / "followup_eda" / "ecd_by_procedure.png"))

## 3. Survival: graft failure (ECD < 500) and low ECD (< 1000)

In [ ]:
display(Image(OUTPUTS / "followup_eda" / "km_by_procedure.png"))
for endpoint, e in res["survival"].items():
    print(endpoint, e["overall"])
    display(pd.DataFrame(e["logrank"]).T)

## 4. Visit attendance and rejection

In [ ]:
display(pd.DataFrame(res["visit_attendance"]))
display(Image(OUTPUTS / "followup_eda" / "rejection_effect.png"))

**What this means for the next steps:** irregular visits and censoring → survival models and gap-tolerant inputs; procedure, indication and rejection change both trajectory and risk; split by graft.